In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip install transformers datasets rouge-score --quiet


In [ ]:
from datasets import load_dataset

# CHANGE THIS PATH to your file
data_path = "/content/drive/MyDrive/biomedical_text_generation/data/unseen/sum_unseen.jsonl"

dataset_dict = load_dataset(
    "json",
    data_files={"test": data_path}
)

test_dataset = dataset_dict["test"]

print("Dataset loaded successfully")
print("Number of examples:", len(test_dataset))
print("\nExample record:")
print(test_dataset[0])


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# CHANGE MODEL HERE to compare models
# model_checkpoint = "GanjinZero/biobart-v2-base"
#model_checkpoint = "QizhiPei/biot5-base"
# model_checkpoint = "GanjinZero/biobart-base"
model_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biot5_sum_final"
#model_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biov2bart_sum_final"


tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_checkpoint
).to("cuda")

model.eval()

print("Model loaded:", model_checkpoint)


In [ ]:
def generate_summary(input_text,
                     max_input_length=512,
                     max_new_tokens=256,
                     num_beams=4):

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    ).to("cuda")

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [ ]:
# Change this number each time
index = 3775

input_text = test_dataset[index]["input"]
target_text = test_dataset[index]["target"]

generated_text = generate_summary(input_text)

print("\n" + "="*100)
print("MODEL:", model_checkpoint)
print("INDEX:", index)
print("="*100)

print("\nINPUT:\n")
print(input_text)

print("\nTARGET (REFERENCE):\n")
print(target_text)

print("\nGENERATED (MODEL OUTPUT):\n")
print(generated_text)

print("\n" + "="*100)


In [ ]:
# Change this number each time
index = 3775

input_text = "summarize: " + test_dataset[index]["input"]
target_text = test_dataset[index]["target"]

generated_text = generate_summary(input_text)

print("\n" + "="*100)
print("MODEL:", model_checkpoint)
print("INDEX:", index)
print("="*100)

print("\nINPUT:\n")
print(input_text)

print("\nTARGET (REFERENCE):\n")
print(target_text)

print("\nGENERATED (MODEL OUTPUT):\n")
print(generated_text)

print("\n" + "="*100)


In [ ]:
import json

output_path = "/content/drive/MyDrive/biomedical_text_generation/test_predictions.json"

results = []

for i in range(len(test_dataset)):

    input_text = test_dataset[i]["input"]
    target_text = test_dataset[i]["target"]

    generated_text = generate_summary(input_text)

    results.append({
        "input": input_text,
        "target": target_text,
        "generated": generated_text
    })

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to:", output_path)
